# SAM3 backfill — the 115 SAM3-absent aug120 clips

**What this is.** `aug120_pipeline.py` never passed `--n` to `ph0_sam3.py`
(default 4), so **115 of 201** aug120 clips have **no SAM3 record** — every
`batch_*/sam3/sam3.json` holds exactly 4 clips while `SAM3_RC=0` read as full
coverage. Root cause + accounting: `TanitAD Research Hub/Data Engineering/
Implementation/incoming/2026-08-15-aug120-fusion/AUG120_FUSION_RESULT.md` §3.
This notebook re-runs the SAM3 leg for exactly those clips on the free-Colab
T4 (~30 GPU-min), **banking per clip** so a session death costs one clip,
never the run.

**Deliberately the first workload:** small, known shape — it validates the
whole loop (auth → pull → GPU → bank → far-side verify → resume) before the
label lab spends GPU time.

**Rules built in:** the gap is derived from the fused **records'** own
`perception.absent` marker (count records, not files — C18), cross-checked
against two more far-side sources; every push is far-side verified by byte
round-trip; restart = re-run all cells (the far-side listing resumes).

Smoke mode (`S2_SMOKE=1`): CPU-only plumbing test — stubs the GPU leg, banks
to `Sayood/tanitad-s2-lab/smoke/` instead of the production label repo.

In [ ]:
# --- parameters -------------------------------------------------------------
import os
SMOKE = os.environ.get('S2_SMOKE', '0') == '1'   # CPU plumbing test (stub GPU)
N_LIMIT = int(os.environ.get('S2_N', '0')) or None      # None = the whole gap
BATCH = int(os.environ.get('S2_BATCH', '12'))    # shards per pull (~36 MB each)
FRAME_STRIDE = 8                                  # ph0_sam3 default
GAP_LIMIT = int(os.environ.get('S2_GAP_LIMIT', '0')) or None
if SMOKE and GAP_LIMIT is None:
    GAP_LIMIT = 12                                # smoke: first K records only
if SMOKE and N_LIMIT is None:
    N_LIMIT = 1
print(f'SMOKE={SMOKE} N_LIMIT={N_LIMIT} BATCH={BATCH} GAP_LIMIT={GAP_LIMIT}')

In [ ]:
# --- Drive mount + repo imports (this repo IS the PI's Drive) ---------------
import json, sys, time
from pathlib import Path
try:
    import s2_lab_lib as L
except ImportError:                    # bare Colab: mount Drive, then import
    from google.colab import drive
    drive.mount('/content/drive')
    sys.path.insert(0, '/content/drive/MyDrive/SayBouBase/raw/Projects/'
                       'TanitAD/colab')
    import s2_lab_lib as L
ROOT = L.add_stack_paths()
print('repo root:', ROOT)
L.pip_install_colab(SMOKE)             # no-op off Colab / in smoke
import s2_schema
print('schema:', s2_schema.SCHEMA_VERSION,
      '| v6 drift:', s2_schema.check_v6_drift())

In [ ]:
# --- auth + bank target -----------------------------------------------------
# Token: Colab Secret HF_TOKEN (key icon, left sidebar) — never printed.
api = L.hf_api()
WORK = Path('/content/backfill') if L.in_colab() else \
    ROOT / 'colab' / '_smoke_work' / 'backfill'
WORK.mkdir(parents=True, exist_ok=True)
# smoke banks to the lab repo's smoke/ prefix, NEVER the production label repo
BANK_REPO = L.DS_LAB if SMOKE else L.DS_LABELS
BANK_PREFIX = (L.SMOKE_PREFIX + 'sam3_backfill/') if SMOKE else \
    L.BACKFILL_PREFIX
L.ensure_repo(api, BANK_REPO)
print(f'banking to {BANK_REPO}/{BANK_PREFIX} (far-side verified per clip)')

In [ ]:
# --- the gap, from the RECORDS (C18), cross-checked -------------------------
gap = L.derive_sam3_gap(api, limit=GAP_LIMIT)
L.cross_check_gap(api, gap, partial=GAP_LIMIT is not None)
if GAP_LIMIT is None:
    L.check_gap_fixture(gap, ROOT)     # loud diff vs the banked dev-box list
todo_all = gap['absent']
print(f'SAM3-absent clips: {len(todo_all)} '
      f'(of {gap["n_records_checked"]} records checked)')

In [ ]:
# --- resume: find what is done, then continue -------------------------------
done = L.done_set(api, BANK_REPO, BANK_PREFIX)
todo = [c for c in todo_all if c not in done]
if N_LIMIT:
    todo = todo[:N_LIMIT]
print(f'far side already holds {len(done)} -> this run: {len(todo)} clips')

In [ ]:
# --- v2 records (the B3 sign boxes SAM3 cross-checks) -----------------------
v2_by = L.load_v2_records(api, set(todo)) if todo else {}

In [ ]:
# --- video locations + Alpamayo join (real runs only) -----------------------
loc, REC_PQ = {}, None
if todo and not SMOKE:
    loc = L.w120_locations(api)
    missing_video = [c for c in todo if c not in loc]
    assert not missing_video, (f'{len(missing_video)} gap clips lack w120 '
                               f'shards: {missing_video[:3]}')
    REC_PQ = str(WORK / 'records.parquet')
    if not Path(REC_PQ).exists():
        import shutil
        shutil.copyfile(L.hf_download(L.DS_ALP, 'records.parquet'), REC_PQ)
    print(f'w120 shards located for all {len(todo)} clips')

In [ ]:
# --- the SAM3 leg: pull batch -> bridge -> detect -> BANK PER CLIP ----------
import shutil
proc = None
if todo and not SMOKE:
    proc, _meta = L.load_sam3()
    L.gpu_mem_report('sam3 load')
t_start, n_banked = time.time(), 0
for b0 in range(0, len(todo), BATCH):
    batch = todo[b0:b0 + BATCH]
    bwork = WORK / f'b{b0:05d}'
    if SMOKE:
        frames_by = {c: L.stub_frames() for c in batch}
    else:
        L.bridge_batch(batch, loc, REC_PQ, bwork)   # pulls + DELETES shards
        import ph0_pilot
        frames_by = {c: ph0_pilot.sample_clip_frames(
            str(bwork / 'videos' / f'{c}.mp4'), t0_s=8.0)[0] for c in batch}
    for cid in batch:
        rec = (L.stub_sam3_record(cid) if SMOKE else
               L.sam3_leg(proc, frames_by[cid], v2_by[cid],
                          frame_stride=FRAME_STRIDE))
        # ⛔ the count is EXPLICIT, always — the --n default of 4 is the
        # measured root cause of this very gap (AUG120_FUSION_RESULT.md §3)
        rec['_n_explicit'] = len(batch)
        sz = L.bank_json(api, BANK_REPO, f'{BANK_PREFIX}{cid}.json', rec)
        n_banked += 1
        print(f'[bank] {n_banked}/{len(todo)} {cid[:8]} {sz} B '
              'far-side-verified', flush=True)
    L.gpu_mem_report(f'after batch b{b0:05d}')
    shutil.rmtree(bwork, ignore_errors=True)
print(f'BANKED {n_banked} clips in {time.time() - t_start:.0f}s')

In [ ]:
# --- final accounting + run manifest + the escalation -----------------------
final_done = L.done_set(api, BANK_REPO, BANK_PREFIX, verify_sample=False)
resid = [c for c in todo_all if c not in final_done]
L.run_manifest(api, BANK_REPO, BANK_PREFIX, 'sam3-backfill', {
    'smoke': SMOKE, 'todo_this_run': len(todo), 'banked_this_run': n_banked,
    'far_side_done_now': len(final_done), 'residual_gap': len(resid),
    'frame_stride': FRAME_STRIDE,
    'n_rule': 'clip count explicit always (aug120 --n root cause)',
    'evidence_class': 'SMOKE-STUB' if SMOKE else 'MEASURED'})
print(f'far side now holds {len(final_done)} · residual gap {len(resid)}')
print('NEXT (escalated here, not buried): re-fuse the backfilled clips — '
      'the 115 fused records still carry perception.absent and must be '
      're-emitted. Owner: aug120-fusion package '
      '(AUG120_FUSION_RESULT.md §9 items 1-2, fuser resumes per clip).')
print('BACKFILL_DONE')